# DQN Atari Pong: Hyperparameter Tuning Experiments

## Overview

This notebook trains a Deep Q-Network (DQN) agent to play Atari Pong using
Stable Baselines 3 and Gymnasium. We run 10 experiments, each with a different
hyperparameter combination, then save all results to queen_experiments.csv.

### What is DQN?
DQN combines Q-learning with a deep neural network. Instead of a Q-table, a neural
network estimates Q(s,a), the expected future reward for taking action a in state s.
Two key tricks make it stable:
- Replay Buffer: stores past experiences and samples them randomly to break correlations
- Target Network: a slowly-updated copy of the network to stabilize Q-value targets

### Hyperparameters Being Tuned
| Hyperparameter | What it controls |
|---|---|
| lr | How fast the network updates its weights |
| gamma | How much future rewards are valued vs immediate rewards |
| batch_size | Number of experiences sampled per training update |
| exploration_fraction | How quickly epsilon decays from 1.0 to its final value |
| exploration_final_eps | Minimum exploration rate (agent never goes fully greedy) |
| policy | CnnPolicy (spatial pixel reading) vs MlpPolicy (flat vector) |

## 1. Setup

In [ ]:
!pip install torch -q
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — go to Runtime > Change runtime type")

GPU available: True
Device: Tesla T4


In [ ]:
!pip install stable-baselines3[extra] gymnasium[atari] ale-py autorom opencv-python-headless --quiet
!AutoROM --accept-license --quiet
print("All packages installed and Atari ROMs downloaded.")

AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.
All packages installed and Atari ROMs downloaded.


In [ ]:
import gymnasium as gym
import ale_py
import warnings
import logging
logging.getLogger('ale_py').setLevel(logging.ERROR)

env = gym.make('PongNoFrameskip-v4')
obs, info = env.reset()
print("Environment loaded successfully.")
print(f"  Observation shape : {obs.shape}")
print(f"  Action space      : {env.action_space}")
print(f"  Number of actions : {env.action_space.n}")
env.close()

Environment loaded successfully.
  Observation shape : (210, 160, 3)
  Action space      : Discrete(6)
  Number of actions : 6


## 2. Upload the Scripts

Upload the three Python files:
- train.py — Training script
- play.py — Playback and evaluation script
- env_utils.py — Shared environment utilities imported by both scripts

In [ ]:
import os, shutil, glob

for f in ['train.py', 'play.py', 'env_utils.py']:
    found = glob.glob(f'/kaggle/input/**/{f}', recursive=True)
    if found:
        shutil.copy(found[0], f'/kaggle/working/{f}')
        print(f"✅ {f} copied")
    else:
        print(f"❌ {f} not found — check upload")

os.chdir('/kaggle/working')
print("\nReady to train!")

✅ train.py copied
✅ play.py copied
✅ env_utils.py copied

Ready to train!


In [ ]:
import os

required = ['train.py', 'play.py', 'env_utils.py']
all_present = True

for f in required:
    status = "OK" if os.path.exists(f) else "MISSING"
    print(f"  [{status}]  {f}")
    if not os.path.exists(f):
        all_present = False

print()
print("Ready to train." if all_present else "Re-upload the missing files before continuing.")

  [OK]  train.py
  [OK]  play.py
  [OK]  env_utils.py

Ready to train.


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('ale_py').setLevel(logging.ERROR)

with open('env_utils.py', 'w') as f:
    f.write('''import subprocess
import sys
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, VecTransposeImage

ENV_ID = "PongNoFrameskip-v4"
N_ENVS = 1
N_STACK = 4
SEED = 0

def check_requirements():
    required = ["stable_baselines3", "gymnasium", "ale_py"]
    for pkg in required:
        try:
            __import__(pkg)
        except ImportError:
            print(f"WARNING: {pkg} not installed")

def make_env(seed=None, render_mode=None, transpose_image=False):
    if seed is None:
        seed = SEED
    env_kwargs = {}
    if render_mode is not None:
        env_kwargs["render_mode"] = render_mode
    env = make_atari_env(ENV_ID, n_envs=N_ENVS, seed=seed, env_kwargs=env_kwargs if env_kwargs else None)
    env = VecFrameStack(env, n_stack=N_STACK)
    if transpose_image:
        env = VecTransposeImage(env)
    return env
''')
print("env_utils.py written.")

env_utils.py written.


## 3. Run Experiments

**This is the only cell you need to edit.**
All 10 hyperparameter combinations for Queen are pre-filled below.
Just run this cell and the training cell after it.

In [ ]:
import csv
import os
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy
from env_utils import make_env, check_requirements
from pathlib import Path
from datetime import datetime

check_requirements()

member_name = "Queen"

experiments = [
    {"experiment_name": "queen_exp1",  "lr": 1e-4,  "gamma": 0.99,  "batch_size": 32,  "exploration_fraction": 0.10, "exploration_initial_eps": 1.0, "exploration_final_eps": 0.01,  "total_timesteps": 100000,  "policy": "CnnPolicy", "buffer_size": 100000},
    {"experiment_name": "queen_exp2",  "lr": 1e-4,  "gamma": 0.99,  "batch_size": 16,  "exploration_fraction": 0.10, "exploration_initial_eps": 1.0, "exploration_final_eps": 0.01,  "total_timesteps": 100000,  "policy": "CnnPolicy", "buffer_size": 100000},
    {"experiment_name": "queen_exp3",  "lr": 1e-4,  "gamma": 0.99,  "batch_size": 128, "exploration_fraction": 0.10, "exploration_initial_eps": 1.0, "exploration_final_eps": 0.01,  "total_timesteps": 100000,  "policy": "CnnPolicy", "buffer_size": 100000},
    {"experiment_name": "queen_exp4",  "lr": 1e-4,  "gamma": 0.99,  "batch_size": 256, "exploration_fraction": 0.10, "exploration_initial_eps": 1.0, "exploration_final_eps": 0.01,  "total_timesteps": 100000,  "policy": "CnnPolicy", "buffer_size": 100000},
    {"experiment_name": "queen_exp5",  "lr": 5e-5,  "gamma": 0.99,  "batch_size": 64,  "exploration_fraction": 0.15, "exploration_initial_eps": 1.0, "exploration_final_eps": 0.05,  "total_timesteps": 100000,  "policy": "CnnPolicy", "buffer_size": 100000},
    {"experiment_name": "queen_exp6",  "lr": 2e-4,  "gamma": 0.95,  "batch_size": 32,  "exploration_fraction": 0.20, "exploration_initial_eps": 1.0, "exploration_final_eps": 0.01,  "total_timesteps": 300000, "policy": "CnnPolicy", "buffer_size": 100000},
    {"experiment_name": "queen_exp7",  "lr": 1e-4,  "gamma": 0.99,  "batch_size": 64,  "exploration_fraction": 0.25, "exploration_initial_eps": 1.0, "exploration_final_eps": 0.01,  "total_timesteps": 300000, "policy": "CnnPolicy", "buffer_size": 100000},
    {"experiment_name": "queen_exp8",  "lr": 5e-4,  "gamma": 0.90,  "batch_size": 128, "exploration_fraction": 0.10, "exploration_initial_eps": 1.0, "exploration_final_eps": 0.05,  "total_timesteps": 300000, "policy": "CnnPolicy", "buffer_size": 100000},
    {"experiment_name": "queen_exp9",  "lr": 1e-4,  "gamma": 0.999, "batch_size": 16,  "exploration_fraction": 0.10, "exploration_initial_eps": 1.0, "exploration_final_eps": 0.001, "total_timesteps": 300000, "policy": "CnnPolicy", "buffer_size": 100000},
    {"experiment_name": "queen_exp10", "lr": 2e-4,  "gamma": 0.99,  "batch_size": 256, "exploration_fraction": 0.05, "exploration_initial_eps": 1.0, "exploration_final_eps": 0.01,  "total_timesteps": 300000, "policy": "CnnPolicy", "buffer_size": 100000},
]

os.makedirs('results', exist_ok=True)
csv_path = 'results/experiments.csv'
fieldnames = ['member_name','experiment_name','policy','lr','gamma','batch_size',
              'exploration_fraction','exploration_initial_eps','exploration_final_eps',
              'total_timesteps','mean_reward','std_reward','timestamp']

completed = set()
if os.path.exists(csv_path):
    with open(csv_path) as f:
        for row in csv.DictReader(f):
            completed.add(row['experiment_name'])

for i, exp in enumerate(experiments, 1):
    if exp['experiment_name'] in completed:
        print(f"\nSkipping {i}/{len(experiments)}: {exp['experiment_name']} (already done)")
        continue

    print(f"\n{'='*55}")
    print(f"Starting experiment {i}/{len(experiments)}: {exp['experiment_name']}")
    print(f"  lr={exp['lr']} gamma={exp['gamma']} batch={exp['batch_size']}")
    print(f"  expl_fraction={exp['exploration_fraction']} final_eps={exp['exploration_final_eps']}")
    print(f"  policy={exp['policy']} timesteps={exp['total_timesteps']:,}")
    print(f"{'='*55}")

    use_cnn = exp['policy'] == 'CnnPolicy'
    env = make_env(seed=0)
    eval_env = make_env(seed=42, transpose_image=use_cnn)

    model = DQN(
        exp['policy'], env,
        learning_rate=exp['lr'],
        gamma=exp['gamma'],
        batch_size=exp['batch_size'],
        exploration_fraction=exp['exploration_fraction'],
        exploration_initial_eps=exp['exploration_initial_eps'],
        exploration_final_eps=exp['exploration_final_eps'],
        buffer_size=exp['buffer_size'],
        tensorboard_log='runs/',
        verbose=1,
    )

    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path=f"best_model_{exp['experiment_name']}",
        eval_freq=25000,
        deterministic=True,
        render=False,
    )

    model.learn(total_timesteps=exp['total_timesteps'], callback=eval_cb,
                tb_log_name=exp['experiment_name'])
    model.save('dqn_model')

    mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)

    write_header = not os.path.exists(csv_path)
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow({
            'member_name': member_name,
            'experiment_name': exp['experiment_name'],
            'policy': exp['policy'],
            'lr': exp['lr'],
            'gamma': exp['gamma'],
            'batch_size': exp['batch_size'],
            'exploration_fraction': exp['exploration_fraction'],
            'exploration_initial_eps': exp['exploration_initial_eps'],
            'exploration_final_eps': exp['exploration_final_eps'],
            'total_timesteps': exp['total_timesteps'],
            'mean_reward': mean_reward,
            'std_reward': std_reward,
            'timestamp': datetime.now().isoformat(),
        })

    print(f"Training complete")
    print(f"Member: {member_name}")
    print(f"Experiment: {exp['experiment_name']}")
    print(f"Policy: {exp['policy']}")
    print(f"Mean reward: {mean_reward:.2f}")
    print(f"Std reward: {std_reward:.2f}")
    print(f"Results appended to {csv_path}")

    env.close()
    eval_env.close()

print("\nAll experiments complete.")

E0000 00:00:1774432851.304869      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774432851.367964      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774432851.894756      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774432851.894800      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774432851.894803      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774432851.894805      55 computation_placer.cc:177] computation placer already registered. Please check linka


Starting experiment 1/10: queen_exp1
  lr=0.0001 gamma=0.99 batch=32
  expl_fraction=0.1 final_eps=0.01
  policy=CnnPolicy timesteps=100,000
Using cuda device
Wrapping the env in a VecTransposeImage.
Logging to runs/queen_exp1_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 4.01e+03 |
|    ep_rew_mean      | -20.5    |
|    exploration_rate | 0.604    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 317      |
|    time_elapsed     | 12       |
|    total_timesteps  | 3995     |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.000236 |
|    n_updates        | 973      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 3.77e+03 |
|    ep_rew_mean      | -20.5    |
|    exploration_rate | 0.257    |
| time/               |          |
|    episodes         | 8        |


## 4. Results

All experiments append their results to results/experiments.csv automatically.
The cell below copies that file to queen_experiments.csv, loads it, sorts by
mean reward, and prints the final ranked table.

In [ ]:
import os
import shutil
import pandas as pd

src = 'results/experiments.csv'
dst = 'results/queen_experiments.csv'

if os.path.exists(src):
    shutil.copy(src, dst)
    print(f"Results saved to {dst}")
else:
    print("No results file found — make sure at least one experiment completed successfully.")
    import sys; sys.exit()

df = pd.read_csv(dst)
df_sorted = df.sort_values('mean_reward', ascending=False).reset_index(drop=True)

cols = [
    'experiment_name', 'policy', 'lr', 'gamma', 'batch_size',
    'exploration_initial_eps', 'exploration_final_eps',
    'exploration_fraction', 'mean_reward', 'std_reward'
]

print()
print('=' * 80)
print('ALL EXPERIMENTS — Ranked best to worst')
print('=' * 80)
print(df_sorted[cols].to_string(index=False))
print()
print(f"Best  : {df_sorted.iloc[0]['experiment_name']}  ->  Mean Reward: {df_sorted.iloc[0]['mean_reward']:.2f}")
print(f"Worst : {df_sorted.iloc[-1]['experiment_name']}  ->  Mean Reward: {df_sorted.iloc[-1]['mean_reward']:.2f}")

Results saved to results/queen_experiments.csv

ALL EXPERIMENTS — Ranked best to worst
experiment_name    policy      lr  gamma  batch_size  exploration_initial_eps  exploration_final_eps  exploration_fraction  mean_reward  std_reward
     queen_exp7 CnnPolicy 0.00010  0.990          64                      1.0                  0.010                  0.25        -17.0    1.264911
     queen_exp5 CnnPolicy 0.00005  0.990          64                      1.0                  0.050                  0.15        -20.4    0.489898
     queen_exp2 CnnPolicy 0.00010  0.990          16                      1.0                  0.010                  0.10        -21.0    0.000000
     queen_exp1 CnnPolicy 0.00010  0.990          32                      1.0                  0.010                  0.10        -21.0    0.000000
     queen_exp4 CnnPolicy 0.00010  0.990         256                      1.0                  0.010                  0.10        -21.0    0.000000
     queen_exp3 CnnPolicy

In [ ]:
import shutil
shutil.copy('results/experiments.csv', '/kaggle/working/queen_experiments.csv')
print("✅ Results saved!")
print("Click Output tab on the right to download queen_experiments.csv")

✅ Results saved!
Click Output tab on the right to download queen_experiments.csv
